# Diffusion-Based Novel View Synthesis — Colab Runner

**What this notebook does:**
1. Mounts Google Drive to load your pre-selected input frames
2. Installs Zero123++ (diffusion model for single-image NVS)
3. Generates 6 novel views per input image (we keep 4)
4. Creates comparison grids (input + novel views)
5. Saves all outputs back to Google Drive

**Before running:**
- Runtime → Change runtime type → **T4 GPU** (or better)
- Upload your `diffusion_nvs/inputs/` folder to Google Drive at:
  `My Drive/NVS/diffusion_nvs/inputs/`
- Adjust `DRIVE_ROOT` below if you placed it elsewhere

## 0. Mount Drive & configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

# --- CONFIGURE THESE PATHS ---
DRIVE_ROOT   = Path('/content/drive/MyDrive/NVS')          # where you uploaded the project
INPUT_DIR    = DRIVE_ROOT / 'diffusion_nvs' / 'inputs'     # pre-selected frames
OUTPUT_DIR   = DRIVE_ROOT / 'diffusion_nvs' / 'outputs' / 'novel_views'
GRID_DIR     = DRIVE_ROOT / 'diffusion_nvs' / 'outputs' / 'grids'
# ------------------------------

N_VIEWS_PER_IMAGE = 4   # how many of the 6 generated views to keep
BATCH_SIZE        = 1   # images processed at a time (keep 1 for T4)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GRID_DIR.mkdir(parents=True, exist_ok=True)

images = sorted(INPUT_DIR.glob('*.jpg')) + sorted(INPUT_DIR.glob('*.png'))
print(f'Found {len(images)} input images in {INPUT_DIR}')
assert len(images) > 0, f'No images found — check INPUT_DIR path: {INPUT_DIR}'

## 1. Install dependencies

In [ ]:
%%capture
!pip install --upgrade diffusers[torch] transformers accelerate
!pip install Pillow numpy tqdm imageio matplotlib

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Load Zero123++ pipeline

Zero123++ generates **6 views at once** from a single input image in a 2×3 grid.
We split the grid into individual images and keep the first `N_VIEWS_PER_IMAGE` views.

In [ ]:
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler
from PIL import Image
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32

print('Loading Zero123++ pipeline (~5 GB — takes a few minutes on first run)...')
pipeline = DiffusionPipeline.from_pretrained(
    'sudo-ai/zero123plus-v1.2',
    custom_pipeline='sudo-ai/zero123plus-pipeline',
    torch_dtype=dtype,
)
pipeline.scheduler = EulerAncestralDiscreteScheduler.from_config(
    pipeline.scheduler.config,
    timestep_spacing='trailing',
)
pipeline = pipeline.to(device)
pipeline.enable_attention_slicing()
print('Pipeline ready.')

## 3. Helper functions

In [ ]:
def preprocess_image(image_path: Path, size: int = 320) -> Image.Image:
    """Square-crop, white-background composite, resize."""
    img = Image.open(image_path).convert('RGBA')
    w, h = img.size
    crop = min(w, h)
    img = img.crop(((w - crop) // 2, (h - crop) // 2,
                    (w + crop) // 2, (h + crop) // 2))
    bg = Image.new('RGBA', img.size, (255, 255, 255, 255))
    bg.paste(img, mask=img.split()[3])
    return bg.convert('RGB').resize((size, size), Image.LANCZOS)


def split_zero123plus_grid(grid_img: Image.Image, n_views: int = 6) -> list[Image.Image]:
    """
    Zero123++ outputs a 960x640 image with 6 views arranged as 2 columns × 3 rows.
    Each cell is 320x320.
    Layout (row-major, left-to-right, top-to-bottom):
      [0] [1]
      [2] [3]
      [4] [5]
    Approximate azimuths: 30°, 90°, 150°, 210°, 270°, 330° (fixed, elevation ~20° up)
    """
    w, h = grid_img.size  # 960 x 640
    cols, rows = 2, 3
    cell_w, cell_h = w // cols, h // rows
    views = []
    for r in range(rows):
        for c in range(cols):
            box = (c * cell_w, r * cell_h, (c + 1) * cell_w, (r + 1) * cell_h)
            views.append(grid_img.crop(box))
    return views[:n_views]


# Zero123++ fixed view angles (approximate, elevation ~20° above horizon)
ZERO123PLUS_AZIMUTHS = [30, 90, 150, 210, 270, 330]
ZERO123PLUS_ELEVATION = 20

print('Helper functions defined.')

## 4. Generate novel views

In [ ]:
from tqdm.notebook import tqdm

total_generated = 0

for img_path in tqdm(images, desc='Generating novel views'):
    stem = img_path.stem
    input_img = preprocess_image(img_path, size=320)

    # Save preprocessed input for comparison grids
    input_img.save(OUTPUT_DIR / f'{stem}_input.png')

    # Run Zero123++ inference
    with torch.no_grad():
        result = pipeline(
            input_img,
            num_inference_steps=75,
            guidance_scale=4.0,
        )

    # Split 2×3 grid into individual views
    all_views = split_zero123plus_grid(result.images[0], n_views=6)
    selected_views = all_views[:N_VIEWS_PER_IMAGE]
    selected_azimuths = ZERO123PLUS_AZIMUTHS[:N_VIEWS_PER_IMAGE]

    for i, (view, azim) in enumerate(zip(selected_views, selected_azimuths)):
        fname = f'{stem}_elev+{ZERO123PLUS_ELEVATION}_azim+{azim:03d}.png'
        view.save(OUTPUT_DIR / fname)
        total_generated += 1

print(f'\nGenerated {total_generated} novel views → {OUTPUT_DIR}')

## 5. Create comparison grids

In [ ]:
from PIL import ImageDraw

def make_grid(input_img: Image.Image, novel_views: list[Image.Image],
              labels: list[str], cell_size: int = 320) -> Image.Image:
    """Horizontal strip: input | view0 | view1 | ..."""
    n = 1 + len(novel_views)
    label_h = 36
    grid = Image.new('RGB', (cell_size * n, cell_size + label_h), (255, 255, 255))
    draw = ImageDraw.Draw(grid)

    for i, (img, label) in enumerate(zip([input_img] + novel_views, ['Input'] + labels)):
        img_r = img.resize((cell_size, cell_size), Image.LANCZOS)
        grid.paste(img_r, (i * cell_size, label_h))
        draw.text((i * cell_size + cell_size // 2, 10), label,
                  fill=(0, 0, 0), anchor='mm')
    return grid


input_images = sorted(OUTPUT_DIR.glob('*_input.png'))
grid_count = 0

for input_path in tqdm(input_images, desc='Creating grids'):
    stem = input_path.stem.replace('_input', '')
    novel_paths = sorted(OUTPUT_DIR.glob(f'{stem}_elev*.png'))
    if not novel_paths:
        print(f'  WARNING: no novel views for {stem}')
        continue

    input_img  = Image.open(input_path).convert('RGB')
    novel_imgs = [Image.open(p).convert('RGB') for p in novel_paths]

    labels = []
    for p in novel_paths:
        parts = p.stem.split('_')
        elev = next((x for x in parts if x.startswith('elev')), '')
        azim = next((x for x in parts if x.startswith('azim')), '')
        labels.append(f'{elev}\n{azim}')

    grid = make_grid(input_img, novel_imgs, labels, cell_size=320)
    grid.save(GRID_DIR / f'{stem}_grid.png')
    grid_count += 1

print(f'\n{grid_count} grids saved → {GRID_DIR}')

## 6. Preview a sample grid

In [ ]:
import matplotlib.pyplot as plt

grids = sorted(GRID_DIR.glob('*_grid.png'))
if grids:
    # Show first and last grid as a sanity check
    samples = [grids[0], grids[len(grids) // 2], grids[-1]]
    fig, axes = plt.subplots(len(samples), 1, figsize=(20, 6 * len(samples)))
    if len(samples) == 1:
        axes = [axes]
    for ax, grid_path in zip(axes, samples):
        img = Image.open(grid_path)
        ax.imshow(img)
        ax.set_title(grid_path.name, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No grids found — check previous cells for errors.')

## Done!

Results saved to Google Drive:
- Novel views: `NVS/diffusion_nvs/outputs/novel_views/`
- Comparison grids: `NVS/diffusion_nvs/outputs/grids/`

Download them to your Mac:
```bash
# From project root, after syncing Drive:
cp -r /path/to/drive/NVS/diffusion_nvs/outputs/ diffusion_nvs/outputs/
```